# Enriched network IoCs feed
The `AdvancedActions.enriched_network_iocs_feed` function from the `advanced` RLSDK module queries the Network IOCs feed (TCF-0301) for new network-related indicators and enriches each indicator with a network threat intelligence report on the go. This notebook demonstrates how to use that function and understand what it returns.

### Used Spectra Intelligence classes
- AdvancedActions

### Credentials
Credentials are loaded from a local file instead of being written here in plain text.
To learn how to creat the credentials file, see the **Storing and using the credentials** section in the [README file](./README.md)

In [ ]:
import json
import os
from ReversingLabs.SDK.advanced import AdvancedActions


CREDENTIALS = json.load(open("credentials.json"))
USERNAME = CREDENTIALS.get("ticloud").get("username")
PASSWORD = CREDENTIALS.get("ticloud").get("password")

cwd = os.getcwd()
upper_level = os.path.dirname(cwd)
USER_AGENT = json.load(open(os.path.join(upper_level, "user_agent.json")))["user_agent"]

advanced_actions = AdvancedActions(
    host="https://data.reversinglabs.com",
    username=USERNAME,
    password=PASSWORD,
    user_agent=USER_AGENT
)

### Defining a time period and indicator type
The first thing we need to do is define the time period from which we want to fetch indicators. Since this feed is designed to deliver fresh network IoCs, the way to define the time period is by stating a singe timestamp. The time period for returning indicators will then be from that moment to the present one.
There are two time formats accepted: UNIX timestamp and UTC. For the purpose of this example, we will go with a UNIX timestamp.

In [ ]:
from datetime import datetime, timedelta

seconds = 5
current_time = datetime.now()
older_time = current_time - timedelta(seconds=seconds)
unix_timestamp = str(int(older_time.timestamp()))

response = advanced_actions.enriched_network_iocs_feed(
    time_format="timestamp",
    time_value=unix_timestamp
)

print(json.dumps(response))

The above code defines a timestamp that is 5 seconds older than the current moment and uses it to fetch indicators.
We have to observe two things here:
1. No specific IoC types were defined in the function call so the function will try to return all available types.
2. The time period is very small (last 5 seconds) because the feed returns large numbers of indicators. When paired with enriching each indicator with a network report, the execution time can become significantly long. In case you need to increase the time period, keep that info in mind.

### Using UTC time string and selecting indicator types
Another way to fetch desired indicators is by using a UTC string and defining only specific indicator types.

In [ ]:
seconds = 5
current_time = datetime.now()
older_time = current_time - timedelta(seconds=seconds)
utc_time = older_time.strftime("%Y-%m-%dT%H:%M:%S")

response = advanced_actions.enriched_network_iocs_feed(
    time_format="utc",
    time_value=utc_time,
    ioc_types=["url, domain"]
)

print(json.dumps(response))

The `ioc_types` parameter accepts either a list of strings or `None` as a value. `None` is its default value so it doesn't have to be explicitly passed.
If a specific list of IoC types needs to be defined, keep in mind the following:
- Possible values are "url", "domain" and "ip".
- In the time of writing this notebook, the feed returns only "url" indicators. It will be expanded to support "domain" and "ip" indicators later on.
- If a specific type of indicator is not supported or not present in the defined time period, no results will be returned for it.